# Week 8: NLP & Text Embeddings
## Notebook 06: Subword Tokenization: SentencePiece & Unigram Language Model from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Early NLP used whitespace tokenization: splitting sentences by spaces.
- This creates massive vocabularies ($> 500,000$ words).
- It completely fails on Out-Of-Vocabulary (OOV) words (e.g. typos like "transformerrr" or rare compounds).
- It fails completely on non-segmented languages like Chinese, Japanese, or Thai that do not use spaces!

Google's **SentencePiece** treats input as a raw byte stream, replacing spaces with a meta-symbol (`_`).
Its flagship algorithm, the **Unigram Tokenizer** (used in ALBERT, LLaMA, T5):
- Starts with a **huge initial vocabulary** of all frequent substrings.
- Iteratively computes the probability of each token in the corpus.
- **Prunes the bottom 20% of tokens** that contribute least to overall corpus likelihood until reaching the target vocabulary size!

---

### 2. Mathematical Foundations: Unigram Tokenizer Optimization
Given vocabulary $V = \{x_1, \dots, x_N\}$ with probabilities $p(x)$ where $\sum_{x \in V} p(x) = 1$:

1. **Sentence Likelihood:**
   The likelihood of sentence $X$ is the maximum probability across all possible segmentations $\mathbf{x} = (x_1, \dots, x_m)$:
   $$P(X) = \max_{\mathbf{x} \in S(X)} \prod_{i=1}^m p(x_i)$$
   Finding the optimal segmentation $\mathbf{x}^*$ is solved in $O(L^2)$ time using the **Viterbi Algorithm** (Dynamic Programming).

2. **Vocabulary Pruning Loss:**
   For each token $x \in V$, compute the drop in corpus log-likelihood if $x$ were removed from the vocabulary:
   $$\mathcal{L}_{\text{drop}}(x) = \mathcal{L}(V) - \mathcal{L}(V \setminus \{x\})$$
   Sort tokens by $\mathcal{L}_{\text{drop}}$ and discard the lowest 20%.

In [ ]:
import math

# Step 1: Small Corpus of Words with Frequencies
corpus = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3
}

# Initial seed vocabulary of characters and common subwords
initial_vocab = ["l", "o", "w", "e", "r", "n", "s", "t", "i", "d", "er", "est", "low", "new"]
# Assign uniform initial probabilities
V_probs = {token: 1.0 / len(initial_vocab) for token in initial_vocab}

# Step 2: Implement Viterbi Subword Segmentation
def viterbi_tokenize(word, vocab_probs):
    n = len(word)
    # best_prob[i] stores highest log-prob segmentation of word[:i]
    best_prob = [-math.inf] * (n + 1)
    best_prob[0] = 0.0
    best_token = [None] * (n + 1)
    
    for i in range(1, n + 1):
        for j in range(0, i):
            sub = word[j:i]
            if sub in vocab_probs:
                prob = best_prob[j] + math.log(vocab_probs[sub])
                if prob > best_prob[i]:
                    best_prob[i] = prob
                    best_token[i] = sub
                    
    # Backtrack optimal tokens
    tokens = []
    curr = n
    while curr > 0:
        token = best_token[curr]
        if token is None:
            # Fallback character
            tokens.append(word[curr-1:curr])
            curr -= 1
        else:
            tokens.append(token)
            curr -= len(token)
    return tokens[::-1]

print("--- Viterbi Segmentation with Seed Vocabulary ---")
test_words = ["lower", "newest", "widest", "lowest"]
for w in test_words:
    seg = viterbi_tokenize(w, V_probs)
    print(f"  '{w}' -> {seg}")

### 3. Step-by-Step Code Walkthrough
1. **Dynamic Programming Grid:** `best_prob[i]` tracks optimal prefix likelihood, preventing exponential recursion.
2. **Subword Decomposition:** Words like "lowest" are decomposed into `['low', 'est']`, preserving morphological meaning.
3. **OOV Elimination:** Unknown compound words are segmented down to base characters without throwing out-of-vocabulary errors.

---

### 4. Industry Applications & Trade-offs
- **SentencePiece:** Powers tokenization for Google T5, LLaMA-1/2/3, and Mistral.
- **Language Agnostic:** Treats spaces as regular characters (`_apple`), eliminating custom language-specific regex rules.

---

### 5. Top Interview Questions & Answers
**Q1: How does BPE differ from the Unigram Tokenizer?**
> **Answer:** BPE is a bottom-up greedy algorithm: it starts with characters and iteratively merges the most frequent adjacent pairs. Unigram is a top-down probabilistic algorithm: it starts with a large dictionary and iteratively prunes tokens that contribute least to the overall corpus likelihood.

**Q2: Why does SentencePiece treat whitespace as a character (`_`)?**
> **Answer:** Traditional tokenizers treat whitespace as a delimiter and discard it. This makes lossless de-tokenization impossible (e.g. recovering exact spacing and punctuation). SentencePiece preserves spaces as regular tokens, ensuring 100% reversible (lossless) encoding and decoding.